# Week 10 — Baselines, Model A, and class-imbalance ablation
Validation-only comparison. The held-out test split is not loaded in this notebook. The runner trains baselines before Model A.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_DIR = Path('/content/online-shoppers-nn2026')
if not (REPO_DIR / 'src').exists():
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/azzzzzzzzzzo57-star/online-shoppers-nn2026.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
packages = ['ucimlrepo==0.0.7', 'pandas==2.2.3', 'numpy==1.26.4', 'scikit-learn==1.6.1', 'joblib==1.4.2', 'matplotlib==3.10.0', 'PyYAML==6.0.2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
try:
    import tensorflow as tf
    print('TensorFlow already available:', tf.__version__)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow==2.18.0'], check=True)
if not (REPO_DIR / 'data/raw/online_shoppers.csv').exists():
    subprocess.run([sys.executable, 'data/download_data.py'], check=True)
print('Repository and dependencies ready:', REPO_DIR)


In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
subprocess.run([sys.executable, '-m', 'src.run_week10'], cwd=ROOT, check=True)
results = json.loads((ROOT / 'experiments/week10_baseline_validation.json').read_text(encoding='utf-8'))
model_a = json.loads((ROOT / 'experiments/week10_model_a_validation.json').read_text(encoding='utf-8'))
print('Validation only; test set used:', results['test_set_used'], model_a['test_set_used'])

## Compare both required baselines
The trivial baseline predicts the majority class. Logistic regression is the simple learned baseline. Macro-F1 is the primary metric.

In [ ]:
rows = []
for name, m in results['models'].items():
    rows.append({'model': name, 'accuracy': m['accuracy'], 'macro_f1': m['macro_f1'], 'no_purchase_recall': m['per_class_recall']['no_purchase_0'], 'purchase_recall': m['per_class_recall']['purchase_1'], 'purchase_precision': m['purchase_precision']})
m = model_a['metrics']
rows.append({'model': 'model_a_neural_network', 'accuracy': m['accuracy'], 'macro_f1': m['macro_f1'], 'no_purchase_recall': m['per_class_recall']['no_purchase_0'], 'purchase_recall': m['per_class_recall']['purchase_1'], 'purchase_precision': m['purchase_precision']})
comparison = pd.DataFrame(rows).set_index('model')
display(comparison.round(4))

## Per-class results and confusion matrices
Confusion-matrix rows are true labels and columns are predicted labels; order is class 0 (no purchase), then class 1 (purchase).

In [ ]:
for name, m in results['models'].items():
    print('\\n', name)
    display(pd.DataFrame(m['confusion_matrix_labels_0_then_1'], index=['true_no_purchase','true_purchase'], columns=['pred_no_purchase','pred_purchase']))
    display(pd.DataFrame(m['classification_report']).T.round(4))
print('\\nModel A — neural network')
display(pd.DataFrame(model_a['metrics']['confusion_matrix_labels_0_then_1'], index=['true_no_purchase','true_purchase'], columns=['pred_no_purchase','pred_purchase']))
display(pd.DataFrame(model_a['metrics']['classification_report']).T.round(4))

## Imbalance ablation
Compare the same logistic-regression algorithm with and without `class_weight='balanced'`. The weights are calculated from training labels only. A rise in purchase recall can trade off against no-purchase recall or precision; interpret all metrics together.

In [ ]:
print(results['imbalance_ablation'])
display(comparison.loc[['logistic_regression_unweighted','logistic_regression_class_weight_balanced']].round(4))

## Interpretation
Use the measured values above in the final report. State whether logistic regression improves macro-F1 over the majority baseline, and whether Model A improves over the learned baseline. Explain the purchase/no-purchase recall trade-off from class weighting. Do not invent results or use the test set for model selection.